# Lab 2 : OD Matrices analytics

## Setup of datasets and visualization

In [3]:
%pip install geopandas "folium>=0.12" matplotlib mapclassify 

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install geopandas

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pymongo as pm
from pprint import pprint
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timezone
import geopandas as gpd
from shapely.geometry import box
from geopandas import GeoDataFrame
from shapely.geometry import box
from branca.colormap import linear
from IPython.display import display
import seaborn as sns
import ipywidgets as widgets
from itertools import combinations

In [6]:
client = pm.MongoClient(
 'bigdatadb.polito.it',
 ssl=True,
 authSource='carsharing',
 username='ictts',
 password='Ict4SM22!',
 tlsAllowInvalidCertificates=True
)

In [8]:
db= client['carsharing']
pb_coll = db['ictts_PermanentBookings']
pb_coll_enjoy= db['ictts_enjoy_PermanentBookings']

In [10]:
pipe = [
    {
        "$project": {
            "hour": {"$hour": "$init_date"},  
            "day": {"$dayOfWeek": "$init_date"},
            "init_location": 1,
            "final_location": 1,
        }
    },
    {
        "$match": {
            "day": {"$gte":1,"$lte":6},
            "hour": {"$gte":0,"$lte":12},
             "driving.distance": {"$gt": 0}
        }
    },
    {
        "$count": "tot"
    }
]

pb_coll.aggregate(pipe)

In [6]:
Q= 0.90

def load_wtime(db, coll, carsharing=True):
    if carsharing:
        pipe = [
            {
                "$match": {
                    "driving.duration": {"$gt": 0},
                    "driving.distance": {"$gt": 0}
                }
            },
            {
                "$project": {
                    "_id": 0,
                    "init_time": 1,
                    "final_time": 1,
                    "duration": "$driving.duration",
                    "day": {
                        "$dateToString": {
                            "date": "$init_date",
                            "format": "%Y-%m-%d"
                        }
                    },
                    "week": {"$isoWeek": "$init_date"},
                    "iso_year": {"$isoWeekYear": "$init_date"},
                    "dow": {"$isoDayOfWeek": "$init_date"},
                    "hour": {"$hour": "$init_date"},
                    "vendor": 1,
                    "init_address": 1,
                    "final_address": 1,
                    "plate": 1,
                    "init_loc": 1,
                    "final_loc": 1,
                    "driving": 1,
                    "walking": 1,
                    'init_date':1
                }
            }
        ]

    else:
        raise NotImplementedError(
            "Non-carsharing data processing is not implemented yet."
        )

    collection = db[coll] if isinstance(coll, str) else coll
    df = pd.DataFrame(collection.aggregate(pipe, allowDiskUse=True))

    if df.empty:
        print("Warning: MongoDB query returned no documents.")
        return df

    df["duration_min"] = df["duration"] / 60

    df["driving_distance_km"] = df["driving"].apply(
        lambda x: x["distance"] / 1000
        if isinstance(x, dict) else np.nan
    )

    df["walking_duration_min"] = df["walking"].apply(
        lambda x: x["duration"] / 60
        if isinstance(x, dict) else np.nan
    )

    return df



def best_zone(x, q=Q, log=True, threshold=False, min_duration=1.0):
    x = np.sort(np.asarray(x, dtype=float))
    if threshold:
        x = x[x >= min_duration]
    else:
        x = x[x >= 0]
    n = len(x)
    if n == 0:
        return np.nan, np.nan
    k = int(np.ceil(q * n))
    t = np.log(x) if log else x
    widths = t[k - 1:] - t[:n - k + 1]
    i = np.argmin(widths)
    return x[i], x[i + k - 1]


def keep_best_zone(df, q=Q, col="duration_min", log=True, threshold=False):
    a, b = best_zone(df[col].values, q, log, threshold)
    return df[df[col].between(a, b)].copy(), (a, b)


def get_filtered_data(db,coll,q=Q,carsharing=True,verbose=True,return_extra=False):
    raw = load_wtime(db=db,coll=coll,carsharing=carsharing)

    if raw.empty:
        raise ValueError("No data returned by MongoDB.")

    filtered, zone = keep_best_zone(raw, q=q)

    if verbose:
        a, b = zone
        n0, n1 = len(raw), len(filtered)
        percentage = n1 / n0 if n0 else 0

        print(
            f"zone = [{a:.1f}, {b:.1f}] min "
            f"-> {n1}/{n0} gardés ({percentage:.1%})"
        )

    if return_extra:
        return filtered, {"raw": raw, "zone": zone}

    return filtered

# Q1

For this question we started to look at the data to see if we needed to adapt our filtering from the first lab.

In [9]:
def load_raw(coll, city="Torino"):
    pipe = [
        {
            "$match": {
                "city": city,
               }
        },
        {"$project": {
            "_id": 0,
            "plate": 1,
            "init_date": 1, "final_date": 1,
            "init_time": 1, "final_time": 1,
            "init_loc": 1, "final_loc": 1,
            "drv_dur": "$driving.duration",
            "drv_dist": "$driving.distance",
        }},
    ]
    df = pd.DataFrame(coll.aggregate(pipe, allowDiskUse=True))
    df["rental_min"] = (df["final_time"] - df["init_time"]) / 60
    df["drv_dist_km"] = df["drv_dist"] / 1000
    df["init_date"] = pd.to_datetime(df["init_date"])
    # coordinates to test "start == end"
    df["same_loc"] = df["init_loc"].apply(lambda x: tuple(x["coordinates"])) == \
                     df["final_loc"].apply(lambda x: tuple(x["coordinates"]))
    return df.drop(columns=["init_loc", "final_loc"])

raw = {
    "Car2Go": load_raw(pb_coll),
    "Enjoy":  load_raw(pb_coll_enjoy),
}

def explore(df, name):
    print(f"--- {name} ---")
    print("duration < 1 min      :", (df.rental_min < 1).sum())
    print("duration > 3 h        :", (df.rental_min > 180).sum())
    print("duration > 24 h       :", (df.rental_min > 1440).sum())
    print("distance <= 0         :", (df.drv_dist <= 0).sum())
    print("start == end          :", df.same_loc.sum())
    print(df[["rental_min", "drv_dist_km"]].describe(percentiles=[.01, .05, .5, .95, .99]))

for name, df in raw.items():
    explore(df, name)

--- Car2Go ---
duration < 1 min      : 0
duration > 3 h        : 0
duration > 24 h       : 0
distance <= 0         : 82477
start == end          : 0
          rental_min    drv_dist_km
count  121667.000000  121667.000000
mean       20.889914       1.251278
std         9.849642       2.200316
min         2.100000      -0.001000
1%          4.750000      -0.001000
5%          7.633333      -0.001000
50%        19.433333      -0.001000
95%        39.650000       5.789000
99%        51.750000       8.347680
max        59.983333      33.401000
--- Enjoy ---
duration < 1 min      : 0
duration > 3 h        : 0
duration > 24 h       : 0
distance <= 0         : 61447
start == end          : 0
         rental_min   drv_dist_km
count  97173.000000  97173.000000
mean      23.405766      1.494120
std       10.423864      2.434082
min        3.066667     -0.001000
1%         7.616667     -0.001000
5%         8.466667     -0.001000
50%       20.533333     -0.001000
95%       44.316667      6.278000
9

We can see that there is a lot of duration below 0 so we just filter it in the pipe.\
The only filter needed is a distance one. It cut 67.79% data for the car2go database and 63.23% for the enjoy Database.